# DINOv3 Distillation to YOLO11l-seg

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marjanstoimcev/DINOv3distill/blob/main/dinov3_yolo11_distillation.ipynb)

This notebook demonstrates how to distill knowledge from DINOv3 (ViT-S/16) to YOLO11l-seg using LightlyTrain.

**Workflow:**
1. Install dependencies
2. Upload/prepare the beverage containers dataset
3. Pretrain YOLO11l-seg backbone using DINOv3 distillation (unlabeled images)
4. Fine-tune on labeled instance segmentation task

**Model Comparison:**
- YOLO11l-seg: 27.7M params, 143.0 GFLOPs
- GELAN-C-SEG (YOLOv9): 27.4M params, 144.6 GFLOPs

These are comparable in size!

## 1. Install Dependencies

In [ ]:
# Install lightly-train with ultralytics support
!pip install -q "lightly-train[ultralytics]"

# Verify installation
import lightly_train
print(f"LightlyTrain version: {lightly_train.__version__}")

In [ ]:
# Additional imports
import os
import shutil
from pathlib import Path
from ultralytics import YOLO

# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Dataset Setup

The beverage containers dataset contains 9 classes:
- bottle-glass, bottle-plastic, cup-disposable, cup-handle
- glass-mug, glass-normal, glass-wine, gym bottle, tin can

Dataset splits:
- Train: 4,563 images
- Valid: 1,303 images
- Test: 653 images

**For Colab:** Upload your dataset as a zip file and extract it.

In [ ]:
# Upload dataset zip file (for Colab)
# The dataset should have this structure:
# beverage_containers/
#   ├── train/
#   │   ├── images/
#   │   └── labels/
#   ├── valid/
#   │   ├── images/
#   │   └── labels/
#   ├── test/
#   │   ├── images/
#   │   └── labels/
#   └── data.yaml

from google.colab import files
import zipfile

print("Please upload your dataset zip file...")
uploaded = files.upload()

# Extract the uploaded zip
for filename in uploaded.keys():
    print(f"Extracting {filename}...")
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        zip_ref.extractall('./data')
    print(f"Extracted to ./data")

# List extracted contents
!ls -la ./data

In [ ]:
# Set the dataset path
# Adjust this if your zip extracts to a different folder name
DATASET_PATH = "./data/beverage_containers"

# If running locally instead of Colab, uncomment and use:
# DATASET_PATH = "../atomagentic_autosegmentation/poc_dinov3_segmentation/data/beverage_containers"

dataset_path = Path(DATASET_PATH)
print(f"Using dataset: {dataset_path}")

In [ ]:
# Verify dataset structure
print("Dataset structure:")
print(f"  Root: {dataset_path}")

total_images = 0
for split in ['train', 'valid', 'test']:
    images_path = dataset_path / split / 'images'
    labels_path = dataset_path / split / 'labels'
    
    if images_path.exists():
        num_images = len(list(images_path.glob('*.jpg'))) + len(list(images_path.glob('*.png')))
        total_images += num_images
        print(f"  {split}/images: {num_images} images")
    
    if labels_path.exists():
        num_labels = len(list(labels_path.glob('*.txt')))
        print(f"  {split}/labels: {num_labels} labels")

print(f"\nTotal images: {total_images}")

In [ ]:
# Create a directory with all images for distillation (unlabeled pretraining)
# Distillation only needs images, no labels!

UNLABELED_DIR = Path("./unlabeled_images")
UNLABELED_DIR.mkdir(exist_ok=True)

# Copy all images from train, valid, test to unlabeled directory
# For distillation, we use ALL available images (labels are ignored)

total_copied = 0
for split in ['train', 'valid', 'test']:
    images_path = dataset_path / split / 'images'
    if images_path.exists():
        for img_file in images_path.glob('*'):
            if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']:
                dest = UNLABELED_DIR / img_file.name
                if not dest.exists():
                    shutil.copy2(img_file, dest)
                    total_copied += 1

print(f"Copied {total_copied} images")
print(f"Total images for distillation: {len(list(UNLABELED_DIR.glob('*')))}")

## 3. DINOv3 Distillation to YOLO11l-seg

Now we pretrain the YOLO11l-seg backbone using knowledge distillation from DINOv3.

**Key parameters:**
- `model`: YOLO11l-seg (student model, 27.7M params)
- `teacher`: DINOv3 ViT-S/16 (teacher model)
- `method`: distillation
- `data`: Path to unlabeled images

In [ ]:
# Configure distillation parameters
import lightly_train

# Output directory for pretrained model
OUTPUT_DIR = "./output/dinov3_yolo11l_distill"

# Training configuration
CONFIG = {
    "out": OUTPUT_DIR,
    "data": str(UNLABELED_DIR),
    "model": "ultralytics/yolo11l-seg.yaml",  # Student: YOLO11l-seg (27.7M params)
    "method": "distillation",
    "method_args": {
        "teacher": "dinov3/vits16",  # Teacher: DINOv3 ViT-S/16
    },
    "epochs": 100,  # Adjust based on your compute budget (100-1000 recommended)
    "batch_size": 32,  # Adjust based on GPU memory (32-128 recommended)
}

print("Distillation Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# Run distillation pretraining
# This will take a while depending on your GPU and number of epochs

print("Starting DINOv3 -> YOLO11l-seg distillation...")
print(f"Using {len(list(UNLABELED_DIR.glob('*')))} unlabeled images")
print("="*60)

lightly_train.pretrain(
    out=CONFIG["out"],
    data=CONFIG["data"],
    model=CONFIG["model"],
    method=CONFIG["method"],
    method_args=CONFIG["method_args"],
    epochs=CONFIG["epochs"],
    batch_size=CONFIG["batch_size"],
)

print("="*60)
print("Distillation complete!")

In [ ]:
# Check the output
output_path = Path(OUTPUT_DIR)

print("Output files:")
for f in output_path.rglob('*'):
    if f.is_file():
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.relative_to(output_path)}: {size_mb:.1f} MB")

## 4. Fine-tune for Instance Segmentation

Now we load the pretrained backbone and fine-tune on the labeled beverage containers dataset for instance segmentation with polygon masks.

In [ ]:
# Path to the pretrained model
PRETRAINED_MODEL_PATH = Path(OUTPUT_DIR) / "exported_models" / "exported_last.pt"

# Alternative: if the path structure is different
if not PRETRAINED_MODEL_PATH.exists():
    # Try to find the exported model
    possible_paths = list(Path(OUTPUT_DIR).rglob("*.pt"))
    print("Available .pt files:")
    for p in possible_paths:
        print(f"  {p}")
    if possible_paths:
        PRETRAINED_MODEL_PATH = possible_paths[0]

print(f"Using pretrained model: {PRETRAINED_MODEL_PATH}")

In [ ]:
# Load the pretrained model with Ultralytics
from ultralytics import YOLO

# Load pretrained backbone
model = YOLO(str(PRETRAINED_MODEL_PATH))

print(f"Model loaded: {model.model.__class__.__name__}")
print(f"Task: {model.task}")

In [ ]:
# Check/create data.yaml for fine-tuning
DATA_YAML = dataset_path / "data.yaml"

if DATA_YAML.exists():
    print(f"Using existing data.yaml: {DATA_YAML}")
    with open(DATA_YAML, 'r') as f:
        print(f.read())
else:
    # Create data.yaml if it doesn't exist
    data_yaml_content = f"""# Beverage Containers Dataset
path: {dataset_path.absolute()}
train: train/images
val: valid/images
test: test/images

# Classes
names:
  0: bottle-glass
  1: bottle-plastic
  2: cup-disposable
  3: cup-handle
  4: glass-mug
  5: glass-normal
  6: glass-wine
  7: gym bottle
  8: tin can

nc: 9
"""
    DATA_YAML = Path("data.yaml")
    with open(DATA_YAML, 'w') as f:
        f.write(data_yaml_content)
    print(f"Created data.yaml")
    print(data_yaml_content)

In [ ]:
# Fine-tune for instance segmentation
# The pretrained backbone should give better results than training from scratch

FINETUNE_OUTPUT = "./output/finetune_seg"

results = model.train(
    data=str(DATA_YAML),
    epochs=50,  # Adjust as needed
    imgsz=640,
    batch=16,
    project=FINETUNE_OUTPUT,
    name="yolo11l_seg_beverage",
    patience=10,  # Early stopping
    save=True,
    plots=True,
)

## 5. Evaluate and Compare

Compare the pretrained+finetuned model against training from scratch.

In [ ]:
# Validate the fine-tuned model
metrics = model.val()

print("\nValidation Results (Pretrained + Fine-tuned):")
print(f"  Box mAP50: {metrics.box.map50:.4f}")
print(f"  Box mAP50-95: {metrics.box.map:.4f}")
print(f"  Mask mAP50: {metrics.seg.map50:.4f}")
print(f"  Mask mAP50-95: {metrics.seg.map:.4f}")

In [ ]:
# Optional: Train from scratch for comparison
TRAIN_FROM_SCRATCH = False  # Set to True to compare

if TRAIN_FROM_SCRATCH:
    # Train YOLO11l-seg from scratch (no pretraining)
    model_scratch = YOLO("yolo11l-seg.yaml")
    
    results_scratch = model_scratch.train(
        data=str(DATA_YAML),
        epochs=50,
        imgsz=640,
        batch=16,
        project=FINETUNE_OUTPUT,
        name="yolo11l_seg_scratch",
        patience=10,
    )
    
    metrics_scratch = model_scratch.val()
    
    print("\n" + "="*60)
    print("COMPARISON: Pretrained vs From Scratch")
    print("="*60)
    print(f"  Pretrained Mask mAP50-95: {metrics.seg.map:.4f}")
    print(f"  From Scratch Mask mAP50-95: {metrics_scratch.seg.map:.4f}")
    improvement = (metrics.seg.map - metrics_scratch.seg.map) / metrics_scratch.seg.map * 100
    print(f"  Improvement: {improvement:+.2f}%")

## 6. Inference Demo

In [ ]:
# Run inference on test images
import matplotlib.pyplot as plt
from PIL import Image
import random

# Get some test images
test_images_path = dataset_path / "test" / "images"
test_images = list(test_images_path.glob("*.jpg"))[:5]

if not test_images:
    test_images = list(test_images_path.glob("*.png"))[:5]

print(f"Running inference on {len(test_images)} test images...")

In [ ]:
# Visualize predictions with segmentation masks
fig, axes = plt.subplots(1, min(5, len(test_images)), figsize=(20, 4))

if len(test_images) == 1:
    axes = [axes]

for ax, img_path in zip(axes, test_images):
    # Run prediction
    results = model.predict(str(img_path), verbose=False)
    
    # Plot result with masks
    result_img = results[0].plot()
    
    ax.imshow(result_img[:, :, ::-1])  # BGR to RGB
    ax.axis('off')
    ax.set_title(img_path.name[:20] + "...")

plt.tight_layout()
plt.savefig("inference_results.png", dpi=150, bbox_inches='tight')
plt.show()

print("Results saved to inference_results.png")

## 7. Export Model

In [ ]:
# Export to ONNX for deployment
model.export(format="onnx", imgsz=640, simplify=True)
print("Model exported to ONNX format")

In [ ]:
# Download the trained model (for Colab)
from google.colab import files

# Find the best model
best_model_path = Path(FINETUNE_OUTPUT) / "yolo11l_seg_beverage" / "weights" / "best.pt"

if best_model_path.exists():
    print(f"Downloading {best_model_path}...")
    files.download(str(best_model_path))
else:
    print("Best model not found. Available files:")
    for f in Path(FINETUNE_OUTPUT).rglob("*.pt"):
        print(f"  {f}")

## Summary

This notebook demonstrated:

1. **DINOv3 Distillation**: Using LightlyTrain to distill knowledge from DINOv3 (ViT-S/16) to YOLO11l-seg
2. **Unlabeled Pretraining**: The distillation phase only requires images, no labels
3. **Fine-tuning**: Loading the pretrained backbone and fine-tuning on labeled instance segmentation data
4. **Comparison**: Optionally comparing pretrained vs from-scratch training

**Key Benefits of Distillation:**
- Better feature representations from foundation models
- Can leverage large amounts of unlabeled data
- Typically faster convergence during fine-tuning
- Better generalization, especially on smaller labeled datasets